In [1]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.bq_client import run

ab = run(open("../sql/04_ab_inputs.sql").read())

print(ab.shape)
ab.groupby('grp')[['reached_payment', 'purchased', 'revenue']].mean()

C:\Users\incha\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\google\cloud\bigquery\table.py:2128: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


(5751, 6)


,reached_payment,purchased,revenue
grp,,,
0,1.0,0.764564,63.072389
1,1.0,0.77193,62.874386


In [2]:
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import proportion_effectsize

In [ ]:
baseline = ab['purchased'].mean()
target = baseline * 1.10

effect = proportion_effectsize(target, baseline)
n_per_group = NormalIndPower().solve_power(effect_size = effect, alpha = 0.05, power = 0.8, ratio = 1)

print(f"Baseline checkout-to-purchase rate: {baseline:.1%}")
print(f"Target rate after a 10% relative lift: {target:.1%}")
print(f"Users needed per group: {n_per_group:,.0f}")
print(f"Users available per group: {ab.groupby('grp').size().min():,}")

Baseline checkout-to-purchase rate: 76.8%
Target rate after a 10% relative life: 84.5%
Users needed per group: 411
Users available per group: 2,850


In [4]:
import numpy as np
from statsmodels.stats.proportion import proportions_ztest

In [5]:
rng = np.random.default_rng(42)
baseline = ab['purchased'].mean()
p_flip = (baseline * 1.10 - baseline) / (1-baseline)
ab['sim_purchased'] = ab['purchased']
mask = (ab.grp ==1) & (ab.purchased == 0)
ab.loc[mask, 'sim_purchased'] = (rng.random(mask.sum()) < p_flip).astype(int)
counts = ab.groupby('grp')['sim_purchased'].sum()
nobs = ab.groupby('grp')['sim_purchased'].count()

z,p = proportions_ztest(counts, nobs)

print(ab.groupby('grp')['sim_purchased'].mean())
print(f"z = {z:.2f}, p-value = {p:.4f}")

grp
0    0.764564
1     0.84807
Name: sim_purchased, dtype: Float64
z = -8.01, p-value = 0.0000


In [6]:
buyers = ab[ab.purchased == 1]
rev_per_buyer = buyers['revenue'].mean()
zero_rev = (buyers['revenue'] == 0).sum()

users = len(ab)
months = 3

base_rate = ab['purchased'].mean()
new_rate = base_rate * 1.10

extra_buyers = users * (new_rate - base_rate)
extra_revenue = extra_buyers * rev_per_buyer

print(f"Purchasers with $0 revenue recorded: {zero_rev} of {len(buyers)}")
print(f"Average revenue per purchasing user: ${rev_per_buyer:,.2f}")
print(f"Extra purchasers from a 10% relative lift: {extra_buyers:,.0f} over {months} months")
print(f"Estimated extra revenue: ${extra_revenue:,.0f} total, about ${extra_revenue / months:,.0f} per month")

Purchasers with $0 revenue recorded: 352 of 4418
Average revenue per purchasing user: $81.97
Extra purchasers from a 10% relative lift: 442 over 3 months
Estimated extra revenue: $36,217 total, about $12,072 per month
